# 1初始化模型

1.1加载环境变量并初始化模型（使用init_chat_model/自己指定模型）

In [1]:
# 非支持模型无法自动加载环境遍历，我们需要自己加载环境变量中的base_url和api_key
import os
from langchain.chat_models import init_chat_model

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")
# 初始化模型
model = init_chat_model(
    model="qwen3.6-plus", # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai", # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai，就是默认采用openai的API规范）
    base_url=base_url,
    api_key=api_key
)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

1.2.使用社区导入的方式导入模型，安装依赖langchain_community

In [7]:
from dotenv import load_dotenv
# 用社区依赖引入
from langchain_community.llms import Tongyi

load_dotenv()

model = Tongyi(model="qwen-turbo")

1.3.流式输出与invoke输出

In [8]:
stream = model.stream("你是谁")

In [9]:
for chunk in stream:
    print(chunk,end="",flush=True)

ValueError: request_id: 51871624-acb4-90f4-97a1-98e695da2af1 
 status_code: 401 
 code: InvalidApiKey 
 message: Invalid API-key provided.

In [10]:
from langchain.agents import create_agent

# 1.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

# 2 访问模型

2.1 invoke阻塞调用

In [11]:
from dotenv import load_dotenv
# 用社区依赖引入
from langchain_community.llms import Tongyi

load_dotenv()

model = Tongyi(model="qwen3.6-plus")
# 调用invoke方法
response = model.invoke("月亮的首都是哪里？")

# 查看响应结果
print(response)

ValueError: request_id: 6e1a5a6e-a015-9269-b99d-cc0d4218d2c4 
 status_code: 401 
 code: InvalidApiKey 
 message: Invalid API-key provided.

2.2 stream流式调用

In [8]:
from langchain_core.messages import HumanMessage
# 通过.stream方法实现流式访问
stream = model.stream([HumanMessage(content="月亮的首都是哪里？")])

# stream调用返回的结果是一个generator，方便我们循环获取结果
print(type(stream))
# 如果是aiMessageChunk的话yong .content,如果是str直接打印

# # 遍历stream结果，实时打印AI的回复
for chunk in stream:
    print(chunk, end="", flush=True)

<class 'generator'>
月亮没有“首都”。因为月亮是地球的天然卫星，是一个天体，它本身没有国家、城市或政府，也就没有首都。这个问题可能是一个比喻或幽默的说法，如果是从科幻或文学的角度出发，可以发挥想象力来回答，但在现实中，月亮并没有首都。

如果你是在开玩笑或者用一种诗意的方式提问，欢迎继续分享你的想法！

# 3 智能体调用m

3.1创建智能体指定模型名，由Langchain初始化模型

In [12]:
from langchain.agents import create_agent

# 1.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")
# 2.调用模型，需要传入一个消息列表
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是什么模型"}]
})

print(response)

ValueError: Unable to infer model provider for model='qwen-turbo'. Please specify 'model_provider' directly.

Supported providers: anthropic, anthropic_bedrock, azure_ai, azure_openai, baseten, bedrock, bedrock_converse, cohere, deepseek, fireworks, google_anthropic_vertex, google_genai, google_vertexai, groq, huggingface, ibm, litellm, mistralai, nvidia, ollama, openai, openrouter, perplexity, together, upstage, xai

For help with specific providers, see: https://docs.langchain.com/oss/python/integrations/providers

3.2创建智能体，并使用创建好的model

In [14]:
from langchain.agents import create_agent
from langchain_community.chat_models.tongyi import ChatTongyi

# 1.使用Model类初始化模型
model = ChatTongyi(
    model="qwen-plus"
    # 其它模型参数...
)

# 2.使用初始化好的model创建智能体
agent = create_agent(model=model)
# 2.调用模型，需要传入一个消息列表
response = agent.invoke({
    "messages": [{"role": "user", "content": "月亮的首都是哪里？"}]
})

print(response)

{'messages': [HumanMessage(content='月亮的首都是哪里？', additional_kwargs={}, response_metadata={}, id='0ed0e2cd-a024-46e8-a3d3-645a770f1508'), AIMessage(content='月亮没有首都，因为它不是国家或政治实体，而是一颗天然卫星。\n\n“首都”是人类为国家、主权政体或某些行政区域设立的中央政府所在地，具有法律、政治和行政意义。月球是地球的自然卫星，没有生命、没有政府、没有居民，也没有任何主权归属（根据1967年《外层空间条约》，月球及其他天体不得由国家通过主权主张、使用或占领等方式据为己有）。\n\n不过，如果你是在玩文字游戏或引用某种虚构设定（比如某部科幻小说、动画、童话中把月亮拟人化并设定了“月宫”“广寒宫”作为“月亮的首都”），那可以幽默回应：\n\n🌙 在中国神话中，月亮上有“广寒宫”，住着嫦娥、玉兔和吴刚——若按神话逻辑，“广寒宫”或许可戏称为“月亮的首都” 😄  \n但请注意：这只是文化想象，非科学事实。\n\n总结：  \n✅ 科学上：月球没有首都；  \n✅ 法律上：月球不属于任何国家，不可被占有；  \n✅ 文化上：可借用神话意象进行诗意表达。\n\n需要我为你讲讲《外层空间条约》或中国探月工程（如嫦娥系列）吗？🌕', additional_kwargs={}, response_metadata={'model_name': 'qwen-plus', 'finish_reason': 'stop', 'request_id': '18f10bdd-4850-9fab-aef1-f04591e4e024', 'token_usage': {'input_tokens': 14, 'output_tokens': 281, 'total_tokens': 295, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--019de252-0804-7411-b699-ed03561b4d27-0', tool_calls=[], invalid_tool_calls=[])]}


In [15]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "月亮的首都是哪里？"}]},
    stream_mode="messages"
):
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

月亮没有首都，因为它不是国家或政治实体，而是一颗天然卫星。

“首都”是人类为国家、主权政体设立的行政中心概念，适用于有政府、法律和领土管辖权的政治实体。月球是地球的自然卫星，没有生命、没有政府、没有居民，也不属于任何国家——根据1967年生效的《外层空间条约》（The Outer Space Treaty），月球及其他天体不得由国家通过主权主张、使用或占领等方式据为己有，其探索和利用应为全人类谋福利。

因此，“月亮的首都是哪里？”是一个拟人化或幽默性质的问题，常见于谜语或脑筋急转弯中。有些趣味答案会玩文字游戏，例如：

- “月球（‘月’之‘球’）→ ‘球’谐音‘求’ → ‘求都’？不成立。”  
- 更常见的搞笑回答是：“广寒宫”（中国神话中嫦娥居住的月宫），但这纯属文学想象，非真实行政中心。  
- 或者调侃式回答：“目前暂未建都，正在招标中 🌕” 😄

总结：  
✅ 科学事实：月球无首都，无主权归属；  
✅ 法律依据：《外层空间条约》禁止国家占有月球；  
✅ 文化延伸：神话中有“广寒宫”，但非真实首都。

如您是在解谜语或写创意文案，欢迎补充语境，我可以帮您设计有趣又严谨的回答！ 🌙✨